# Population & Demographic Intelligence Agent

In [1]:
import time
import chromadb
from google import genai
import pandas as pd

In [10]:
client = genai.Client(api_key="Write_Here_Your_Own_API_Address")

In [3]:
df_pop = pd.read_csv("POPH.csv", low_memory=False)
df_pop["date"] = pd.to_datetime(df_pop["date"])
df_pop = df_pop.sort_values("date").reset_index(drop=True)

In [4]:
df_pop["Growth_Rate"] = df_pop["value"].pct_change() * 100

In [5]:
chroma_client = chromadb.PersistentClient(path="./population_agent_db")
collection = chroma_client.get_or_create_collection(name="population_collection")

In [6]:
if collection.count() == 0:
    documents = [
        f"Year: {row['date'].strftime('%Y')} | Total Population: {row['value']} persons | Annual Growth Rate: {row.get('Growth_Rate', 0):.2f}%"
        for _, row in df_pop.iterrows()
    ]
    metadatas = [
        {
            "year": str(row["date"].year),
            "population": float(row["value"]),
            "growth_rate": float(row.get("Growth_Rate", 0)),
        }
        for _, row in df_pop.iterrows()
    ]
    ids = [str(i) for i in range(len(df_pop))]

    collection.add(documents=documents, metadatas=metadatas, ids=ids)
    print(f"Indexed {collection.count()} historical population records into ChromaDB.")
else:
    print(f"Population collection already loaded with {collection.count()} records.")

Indexed 100 historical population records into ChromaDB.


In [7]:
def search_population_database(query: str) -> str:
    """Nüfus veritabanında arama yapar ve ilgili yıllara ait nüfus ve büyüme oranlarını getirir."""
    results = collection.query(query_texts=[query], n_results=5)
    context_blocks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        block = f"Record -> {doc}"
        context_blocks.append(block)
    return "\n".join(context_blocks)

In [8]:
def run_population_agent(user_goal: str, max_retries=3):
    """Kullanıcının demografik hedefini alıp veritabanından bilgi toplayarak Gemini ile analitik rapor üreten ajan."""
    print(f"\n[Agent Goal]: {user_goal}")

    print("[Agent Action]: Querying historical population database...")
    retrieved_data = search_population_database(user_goal)

    prompt = f"""
    You are an autonomous AI Demographic Intelligence Analyst and Economist. 
    Your task is to analyze the historical population records below and answer the user's request with deep insights.
    
    Retrieved Population Data:
    {retrieved_data}
    
    User Request: {user_goal}
    
    Provide a professional, analytical, and structured demographic report with key historical takeaways.
    """

    for attempt in range(max_retries):
        try:
            print(f"[Attempt {attempt+1}] Generating demographic report via Gemini...")
            response = client.models.generate_content(
                model="gemini-3.8-flash", contents=prompt
            )
            return response.text
        except Exception as e:
            print(f"Error encountered: {e}")
            if attempt < max_retries - 1:
                print("Waiting 3 seconds before retrying...")
                time.sleep(3)
            else:
                return "The server is currently busy. Please try running the cell again in a moment."

In [9]:
agent_response = run_population_agent(
    "Analyze the population growth trends and major shifts over the decades."
)
print("\n[Agent Final Output]:\n", agent_response)


[Agent Goal]: Analyze the population growth trends and major shifts over the decades.
[Agent Action]: Querying historical population database...
[Attempt 1] Generating demographic report via Gemini...

[Agent Final Output]:
 # Demographic Intelligence & Economic Analysis Report

**Subject:** 20th-Century Population Growth Trajectories and Structural Transitions  
**Source Dataset:** Historical Population Series (1905–1996)  
**Analyst:** AI Demographic Intelligence Analyst & Economist  

---

### Executive Summary

Between 1905 and 1996, the observed population underwent a **216.4% total expansion**, increasing from 83.8 million to 265.2 million individuals. 

While absolute population grew monotonically across the 91-year timeline, the **annual growth rate exhibited a structural secular decline**, moving from rapid expansion (>2.0% per annum in the early-to-mid 20th century) to a mature demographic regime characterized by sub-1.0% annual growth by the late 1990s. This trajectory refl